# Fusion HMM model-order selection summary figure

This notebook recreates the manuscript planning figure that combines:

- **Panel A:** LOSO-CV K-sweep mean test free energy ± SEM, with the **1-SE threshold** and the **smallest K within 1-SE**
- **Panel B:** Mean matched **state-signature correlation** for **K=3** and **K=5**
- **Panel C:** Median matched **fractional occupancy (FO)** per state across folds for **K=3** and **K=5**

It also saves a compact CSV table used to summarize the K-selection decision.

Edit the file paths in the **User inputs** cell, then run all cells.


In [ ]:

# =========================
# User inputs
# =========================

BASE_DIR = r"/mnt/data"

# K-sweep / selection files
SUMMARY_BYK_TSV = "summary_byK_selected.tsv"
PAIRED_TESTS_TSV = "paired_tests_vs_bestK.tsv"          # optional for the decision table
K_SELECTION_JSON = "K_selection_recommendation.json"

# Stability files
STATE_MATCHING_K03_TSV = "state_matching_scores_K03.tsv"
STATE_MATCHING_K05_TSV = "state_matching_scores_K05.tsv"
FOLD_SUMMARIES_K03_TSV = "fold_summaries_table_matched_K03.tsv"
FOLD_SUMMARIES_K05_TSV = "fold_summaries_table_matched_K05.tsv"

# Figure annotations
ANNOTATE_KS = [3, 5, 9, 12]

# Output files
OUT_FIG = "fusion_hmm_model_selection_summary.png"
OUT_TABLE = "fusion_hmm_K_selection_compact_table.csv"

# Figure settings
FIGSIZE = (12, 10)
DPI = 200


In [ ]:

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def pjoin(*parts):
    return os.path.join(*parts)

summary_byk_path = pjoin(BASE_DIR, SUMMARY_BYK_TSV)
paired_tests_path = pjoin(BASE_DIR, PAIRED_TESTS_TSV)
k_selection_path = pjoin(BASE_DIR, K_SELECTION_JSON)
state_k03_path = pjoin(BASE_DIR, STATE_MATCHING_K03_TSV)
state_k05_path = pjoin(BASE_DIR, STATE_MATCHING_K05_TSV)
fold_k03_path = pjoin(BASE_DIR, FOLD_SUMMARIES_K03_TSV)
fold_k05_path = pjoin(BASE_DIR, FOLD_SUMMARIES_K05_TSV)

sel = pd.read_csv(summary_byk_path, sep="\t")
paired = pd.read_csv(paired_tests_path, sep="\t") if os.path.exists(paired_tests_path) else None
krec = json.load(open(k_selection_path, "r"))
k3_scores = pd.read_csv(state_k03_path, sep="\t")
k5_scores = pd.read_csv(state_k05_path, sep="\t")
k3_fold = pd.read_csv(fold_k03_path, sep="\t")
k5_fold = pd.read_csv(fold_k05_path, sep="\t")

print("Loaded files:")
print("  ", summary_byk_path)
print("  ", k_selection_path)
print("  ", state_k03_path)
print("  ", state_k05_path)
print("  ", fold_k03_path)
print("  ", fold_k05_path)
if paired is not None:
    print("  ", paired_tests_path)

sel.head()


In [ ]:

# Quick check of the main model-selection values

best_k = int(krec["K_best"])
best_mean = float(krec["best_mean_testFE"])
best_sem = float(krec["best_sem_testFE"])
one_se_threshold = float(krec["oneSE_threshold"])
one_se_k = int(krec["K_1se"])
local_minima = krec.get("local_minima", [])

print(f"Best raw K by mean test FE: K={best_k}")
print(f"Best mean test FE: {best_mean:.6f}")
print(f"Best SEM: {best_sem:.6f}")
print(f"1-SE threshold: {one_se_threshold:.6f}")
print(f"Smallest K within 1-SE: K={one_se_k}")
print(f"Local minima: {local_minima}")


In [ ]:

# Helper functions

def state_corr_summary(score_df):
    x = score_df["mean_state_corr"].dropna()
    return {
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)),
        "sem": float(x.std(ddof=1) / np.sqrt(len(x))),
        "median": float(x.median()),
        "n": int(len(x)),
    }

def matched_fo_medians(fold_df):
    fo_cols = [c for c in fold_df.columns if c.startswith("FO_s")]
    return pd.Series({c: fold_df[c].median() for c in fo_cols})

def summarize_k(k, fold_df=None, score_df=None):
    row = sel.loc[sel["K"] == k].iloc[0]
    out = {
        "K": int(k),
        "mean_test_FE": float(row["fe_test_mean"]),
        "SEM_test_FE": float(row["fe_test_sem"]),
        "feasible_frac": float(row["feasible_frac"]),
        "delta_vs_bestK": float(row["fe_test_mean"] - sel.loc[sel["K"] == best_k, "fe_test_mean"].iloc[0]),
        "within_1SE_rule": bool(row["fe_test_mean"] <= one_se_threshold),
        "sweep_FOmax_median": float(row["fo_max_median"]),
        "sweep_n_active_median": float(row["n_active_median"]),
        "sweep_neff_median": float(row["neff_median"]),
    }
    if (fold_df is not None) and (score_df is not None):
        sc = state_corr_summary(score_df)
        out.update({
            "stability_mean_state_corr": sc["mean"],
            "stability_sem_state_corr": sc["sem"],
            "stability_median_state_corr": sc["median"],
            "matched_FOmax_median": float(fold_df["FO_max"].median()),
            "matched_n_active_median": float(fold_df["n_active"].median()),
            "matched_neff_median": float(fold_df["neff"].median()),
        })
    return out


In [ ]:

# Build a compact decision table for K=3, K=5, and K=12

decision_rows = [
    summarize_k(3, k3_fold, k3_scores),
    summarize_k(5, k5_fold, k5_scores),
    summarize_k(12, None, None),
]
decision = pd.DataFrame(decision_rows)

out_table_path = pjoin(BASE_DIR, OUT_TABLE)
decision.to_csv(out_table_path, index=False)

decision


In [ ]:

# Panel C data: median matched FO across folds

k3_fo_medians = matched_fo_medians(k3_fold)
k5_fo_medians = matched_fo_medians(k5_fold)

print("K=3 median matched FO by state:")
display(k3_fo_medians.to_frame("median_FO"))

print("K=5 median matched FO by state:")
display(k5_fo_medians.to_frame("median_FO"))


In [ ]:

# Recreate the summary figure

k3_sc = state_corr_summary(k3_scores)
k5_sc = state_corr_summary(k5_scores)

fig = plt.figure(figsize=FIGSIZE)
gs = GridSpec(2, 2, figure=fig, height_ratios=[1.0, 1.0], width_ratios=[1.2, 1.0])

# -------------------------
# Panel A: K-sweep
# -------------------------
ax1 = fig.add_subplot(gs[0, :])
ax1.errorbar(
    sel["K"],
    sel["fe_test_mean"],
    yerr=sel["fe_test_sem"],
    marker="o",
    linewidth=2,
    capsize=4,
)
ax1.axhline(
    one_se_threshold,
    linestyle="--",
    linewidth=2,
    label=f"1-SE threshold ({one_se_threshold:.2f})",
)
ax1.axvline(
    one_se_k,
    linestyle=":",
    linewidth=2,
    label=f"Smallest within 1-SE: K={one_se_k}",
)

for k in ANNOTATE_KS:
    if k in sel["K"].values:
        y = sel.loc[sel["K"] == k, "fe_test_mean"].iloc[0]
        ax1.annotate(f"K={k}", (k, y), textcoords="offset points", xytext=(0, 8), ha="center")

ax1.set_title("A. LOSO-CV K-sweep: mean test free energy ± SEM")
ax1.set_xlabel("K")
ax1.set_ylabel("Mean test free energy")
ax1.legend(frameon=False)
ax1.grid(False)

# -------------------------
# Panel B: stability bar plot
# -------------------------
ax2 = fig.add_subplot(gs[1, 0])

bars_x = np.array([0, 1])
state_corr_means = [k3_sc["mean"], k5_sc["mean"]]
state_corr_sems = [k3_sc["sem"], k5_sc["sem"]]

ax2.bar(bars_x, state_corr_means, yerr=state_corr_sems, capsize=4)
ax2.set_xticks(bars_x, ["K=3", "K=5"])
ax2.set_ylim(0, 1.02)
ax2.set_ylabel("Mean matched state-signature correlation")
ax2.set_title("B. Stability favors K=3 over K=5")

# -------------------------
# Panel C: matched FO medians
# -------------------------
ax3 = fig.add_subplot(gs[1, 1])

x3 = np.arange(len(k3_fo_medians)) + 1
x5 = np.arange(len(k5_fo_medians)) + 1

ax3.plot(x3, k3_fo_medians.values, marker="o", linewidth=2, label="K=3")
ax3.plot(x5, k5_fo_medians.values, marker="o", linewidth=2, label="K=5")
ax3.set_xticks([1, 2, 3, 4, 5])
ax3.set_xlabel("Matched state index")
ax3.set_ylabel("Median FO across folds")
ax3.set_title("C. K=5 shows one dominant baseline state")
ax3.legend(frameon=False)

fig.suptitle("Fusion HMM model-order selection summary", fontsize=18, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])

out_fig_path = pjoin(BASE_DIR, OUT_FIG)
fig.savefig(out_fig_path, dpi=DPI, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {out_fig_path}")
print(f"Saved table to:  {out_table_path}")


## Notes

This notebook is set up to reproduce the **same summary figure** used for manuscript planning.

For the final manuscript, you may later want a cleaner figure set built from:

- the **K-sweep with 1-SE threshold**, and
- the **K=3 vs K=5 signature-similarity heatmaps**

but this notebook preserves the current composite summary exactly.
